# Course 1, Week 3 — Data Management in PyTorch

- [DeepLearning.AI platform](https://learn.deeplearning.ai/specializations/pytorch-for-deep-learning-professional-certificate/lesson)
- [Week notes](README.md)
- [GitHub issue #3](https://github.com/majorgilles/pytorch_for_deep_learning/issues/3)

**Focus:** Represent datasets and feed repeatable batches through Dataset and DataLoader.

## Video guide

| Video | Covered notebook section |
|---|---|
| [Introduction to Data Pipelines](https://learn.deeplearning.ai/specializations/pytorch-for-deep-learning-professional-certificate/lesson/bvvjkv/introduction-to-data-pipelines) | Introduction to data pipelines |
| Accessing messy data with a custom `Dataset` | Solving data-access problems with a custom `Dataset` |
| Solving image-quality problems with transforms | Solving image-quality problems with transforms |

> The remaining video titles will receive direct DeepLearning.AI links when their lesson URLs are added.


## Download and inspect the raw dataset

The Oxford Flowers data arrives as a compressed image archive and a MATLAB label file. `download_dataset()` stores both under `flower_data/` and returns immediately when the extracted images and labels already exist.

Only paths and metadata need to be in memory during setup. Individual image pixels are loaded later, when the dataset receives an index. Network responses are checked before writing, and archive extraction uses Python's safe data filter.


In [1]:
import os
import tarfile
from typing import cast

import numpy as np
import requests
import torch
from numpy import typing as npt
from PIL import Image
from scipy.io import loadmat
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from tqdm import tqdm


In [2]:
def download_dataset() -> None:
    """
    Downloads and extracts a dataset from remote URLs if not already present locally.

    This function first checks for the existence of the dataset files in a specific
    directory. If the files are not found, it proceeds to download them from
    pre-defined URLs, and then extracts the contents.
    """
    # Define the directory to store the dataset.
    data_dir: str = "flower_data"

    # Define paths for key files and folders.
    image_folder_path: str = os.path.join(data_dir, "jpg")
    labels_file_path: str = os.path.join(data_dir, "imagelabels.mat")
    tgz_path: str = os.path.join(data_dir, "102flowers.tgz")

    # Check if the primary data folder and a key label file already exist.
    if os.path.exists(image_folder_path) and os.path.exists(labels_file_path):
        # Inform the user that the dataset is already available locally.
        print(f"Dataset already exists. Loading locally from '{data_dir}'.")
        # Exit the function since no download is needed.
        return

    # Inform the user that the dataset is not found and the download will start.
    print("Dataset not found locally. Downloading...")

    # Define the URLs for the image archive and the labels file.
    image_url: str = "https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz"
    labels_url: str = (
        "https://www.robots.ox.ac.uk/~vgg/data/flowers/102/imagelabels.mat"
    )

    # Create the target directory for the dataset, if it doesn't already exist.
    os.makedirs(data_dir, exist_ok=True)

    # Announce the start of the image download process.
    print("Downloading images...")
    # Send an HTTP GET request to the image URL, enabling streaming for large files.
    response: requests.Response = requests.get(image_url, stream=True, timeout=60)
    response.raise_for_status()
    # Get the total size of the file from the response headers for the progress bar.
    total_size: int = int(response.headers.get("content-length", 0))

    # Open a local file in binary write mode to save the downloaded archive.
    with open(tgz_path, "wb") as file:
        # Iterate over the response content in chunks with a progress bar.
        file.writelines(
            tqdm(
                # Define the chunk size for iterating over the content.
                response.iter_content(chunk_size=1024),
                # Set the total for the progress bar based on the file size in kilobytes.
                total=total_size // 1024,
            )
        )

    # Announce the start of the file extraction process.
    print("Extracting files...")
    # Open the downloaded tar.gz archive in read mode.
    with tarfile.open(tgz_path, "r:gz") as tar:
        # Extract all contents of the archive into the target directory.
        tar.extractall(data_dir, filter="data")

    # Announce the start of the labels download process.
    print("Downloading labels...")
    # Send an HTTP GET request to the labels URL.
    response = requests.get(labels_url, timeout=60)
    response.raise_for_status()
    # Open a local file in binary write mode to save the labels.
    with open(labels_file_path, "wb") as file:
        # Write the entire content of the response to the file.
        file.write(response.content)

    # Inform the user that the download and extraction are complete.
    print(f"Dataset downloaded and extracted to '{data_dir}'.")

    # create labels_description.txt
    labels_description: list[str] = [
        "pink primrose",
        "hard-leaved pocket orchid",
        "canterbury bells",
        "sweet pea",
        "english marigold",
        "tiger lily",
        "moon orchid",
        "bird of paradise",
        "monkshood",
        "globe thistle",
        "snapdragon",
        "colt's foot",
        "king protea",
        "spear thistle",
        "yellow iris",
        "globe-flower",
        "purple coneflower",
        "peruvian lily",
        "balloon flower",
        "giant white arum lily",
        "fire lily",
        "pincushion flower",
        "fritillary",
        "red ginger",
        "grape hyacinth",
        "corn poppy",
        "prince of wales feathers",
        "stemless gentian",
        "artichoke",
        "sweet william",
        "carnation",
        "garden phlox",
        "love in the mist",
        "mexican aster",
        "alpine sea holly",
        "ruby-lipped cattleya",
        "cape flower",
        "great masterwort",
        "siam tulip",
        "lenten rose",
        "barbeton daisy",
        "daffodil",
        "sword lily",
        "poinsettia",
        "bolero deep blue",
        "wallflower",
        "marigold",
        "buttercup",
        "oxeye daisy",
        "common dandelion",
        "petunia",
        "wild pansy",
        "primula",
        "sunflower",
        "pelargonium",
        "bishop of llandaff",
        "gaura",
        "geranium",
        "orange dahlia",
        "pink-yellow dahlia?",
        "cautleya spicata",
        "japanese anemone",
        "black-eyed susan",
        "silverbush",
        "californian poppy",
        "osteospermum",
        "spring crocus",
        "bearded iris",
        "windflower",
        "tree poppy",
        "gazania",
        "azalea",
        "water lily",
        "rose",
        "thorn apple",
        "morning glory",
        "passion flower",
        "lotus",
        "toad lily",
        "anthurium",
        "frangipani",
        "clematis",
        "hibiscus",
        "columbine",
        "desert-rose",
        "tree mallow",
        "magnolia",
        "cyclamen ",
        "watercress",
        "canna lily",
        "hippeastrum ",
        "bee balm",
        "ball moss",
        "foxglove",
        "bougainvillea",
        "camellia",
        "mallow",
        "mexican petunia",
        "bromelia",
        "blanket flower",
        "trumpet creeper",
        "blackberry lily",
    ]

    with open(
        os.path.join(data_dir, "labels_description.txt"), "w", encoding="utf-8"
    ) as file:
        file.writelines(f"{label}\n" for label in labels_description)

In [3]:
download_dataset()

Dataset already exists. Loading locally from 'flower_data'.


## Define the custom dataset contract

`OxfordFlowersDataset` implements `Dataset[tuple[Image.Image | torch.Tensor, int]]`. Indexing returns an RGB PIL image when no transform is configured, or a tensor when the transform converts the image.

- `__init__()` loads labels but leaves image pixels on disk.
- `__len__()` reports the number of image-label pairs.
- `__getitem__()` maps a zero-based dataset index to a one-based filename and loads only that image.

The `.mat` loader has broad static types, so `cast()` records the narrower `int64` array guaranteed by the explicit NumPy conversion. Raw labels are shifted from $1 \ldots 102$ to class indices $0 \ldots 101$.


In [4]:
class OxfordFlowersDataset(Dataset[tuple[Image.Image | torch.Tensor, int]]):
    """Load Oxford Flowers images lazily from disk.

    Args:
        root_dir: Directory containing ``jpg/`` and ``imagelabels.mat``.
        transform: Optional preprocessing pipeline. Tensor-producing transforms
            conventionally return image shape ``[C, H, W]``.
    """

    def __init__(
        self, root_dir: str, transform: transforms.Compose | None = None
    ) -> None:
        self.root_dir: str = root_dir
        self.img_dir: str = os.path.join(root_dir, "jpg")
        self.transform: transforms.Compose | None = transform

        # Load lightweight metadata now; image pixels remain on disk.
        raw_labels: npt.NDArray[np.int64] = cast(
            npt.NDArray[np.int64],
            np.asarray(
                loadmat(os.path.join(root_dir, "imagelabels.mat"))["labels"][0],
                dtype=np.int64,
            ),
        )
        # PyTorch classification targets use zero-based class indices.
        self.labels: npt.NDArray[np.int64] = raw_labels - 1

    def __len__(self) -> int:
        """Return the number of image-label pairs."""
        return len(self.labels)

    def __getitem__(self, index: int) -> tuple[Image.Image | torch.Tensor, int]:
        """Return one RGB image and its zero-based label.

        Args:
            index: Zero-based sample index.

        Returns:
            An ``(image, label)`` pair. A PIL image uses size ``(W, H)``;
            a transformed tensor conventionally uses shape ``[C, H, W]``.
        """
        # Dataset indices start at 0, while Oxford filenames start at 1.
        image_name: str = f"image_{index + 1:05d}.jpg"
        image_path: str = os.path.join(self.img_dir, image_name)

        # Materialize an RGB copy so the source file can close safely.
        with Image.open(image_path) as source_image:
            image: Image.Image | torch.Tensor = source_image.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        label: int = int(self.labels[index])
        return image, label


### Verify one sample before batching

A one-sample smoke test catches missing files, filename offsets, decoding failures, and invalid labels before those problems are hidden inside a multi-worker `DataLoader`. At this stage the image remains a variably sized PIL image; the next step makes its representation consistent.


In [5]:
dataset: OxfordFlowersDataset = OxfordFlowersDataset(root_dir="./flower_data")
print(f"Total samples: {len(dataset)}")

# Loading one sample exercises filename mapping, image decoding, and label lookup.
image: Image.Image
label: int
image, label = dataset[0]
image, label

Total samples: 8189


(<PIL.Image.Image image mode=RGB size=591x500>, 76)

## Build a consistent image transform

The transform pipeline prepares every flower for the same model input contract:

$$
\text{PIL image} \rightarrow [224, 224] \rightarrow [3, 224, 224] \rightarrow \text{normalized tensor}.
$$

`Resize(256)` preserves aspect ratio by resizing the shorter edge. `CenterCrop(224)` produces a square image, and `ToTensor()` converts channel-last bytes to a channel-first `float32` tensor in $[0, 1]$.

### Why normalize?

For each RGB channel, normalization computes

$$
x' = \frac{x - \mu}{\sigma}.
$$

Centering and scaling the inputs provides a healthy, consistent signal for optimization. This supports stable activations and gradients and helps reduce the risk of vanishing or exploding gradients, but it cannot prevent those problems by itself. The ImageNet statistics below also match the input distribution expected by ImageNet-pretrained models.

Normalization preserves shape $[C, H, W]$ but changes the value range: normalized values are not restricted to $[0, 1]$.


In [6]:
# Standardize each RGB channel with x' = (x - mean) / std.
imagenet_normalize: transforms.Normalize = transforms.Normalize(
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
)

# Geometry runs on the PIL image; normalization runs after tensor conversion.
flower_transform: transforms.Compose = transforms.Compose(
    [
        transforms.Resize(256),  # Resize the shorter edge; preserve aspect ratio.
        transforms.CenterCrop(224),  # PIL size (W, H) becomes (224, 224).
        transforms.ToTensor(),  # [H, W, C] bytes -> [C, H, W] floats in [0, 1].
        imagenet_normalize,  # Keep [C, H, W]; center and scale each channel.
    ]
)


### Inspect each transformation boundary

PIL reports image size as `(W, H)`, while PyTorch tensors conventionally use `[C, H, W]`. Applying each operation separately makes geometry, representation, and scale errors easy to locate. Normalization changes values, not shape.


In [7]:
# Apply each stage separately so type, shape, and scale changes remain visible.
resized_image: Image.Image = transforms.Resize(256)(image)
print(f"After resize, PIL size (W, H): {resized_image.size}")

cropped_image: Image.Image = transforms.CenterCrop(224)(resized_image)
print(f"After crop, PIL size (W, H): {cropped_image.size}")

image_tensor: torch.Tensor = transforms.ToTensor()(cropped_image)
print(f"Tensor shape [C, H, W]: {image_tensor.shape}")
print(f"Before normalization: [{image_tensor.min():.3f}, {image_tensor.max():.3f}]")

normalized_tensor: torch.Tensor = imagenet_normalize(image_tensor)
print(f"Normalized shape [C, H, W]: {normalized_tensor.shape}")
print(f"After normalization: [{normalized_tensor.min():.3f}, {normalized_tensor.max():.3f}]")


After resize, PIL size (W, H): (302, 256)
After crop, PIL size (W, H): (224, 224)
Tensor shape [C, H, W]: torch.Size([3, 224, 224])
Before normalization: [0.000, 1.000]
Normalized shape [C, H, W]: torch.Size([3, 224, 224])
After normalization: [-2.101, 2.640]


In [8]:
transformed_dataset: OxfordFlowersDataset = OxfordFlowersDataset(
    root_dir="./flower_data",
    transform=flower_transform,
)
print(f"Total samples: {len(transformed_dataset)}")

sample_image, sample_label = transformed_dataset[0]
assert isinstance(sample_image, torch.Tensor)
print(f"Sample shape [C, H, W]: {sample_image.shape}")
print(f"Sample label: {sample_label}")


Total samples: 8189
Sample shape [C, H, W]: torch.Size([3, 224, 224])
Sample label: 76


In [9]:
# DataLoader adds a leading batch dimension: [C, H, W] -> [B, C, H, W].
dataloader = DataLoader(transformed_dataset, batch_size=4, shuffle=True)
batch_images, batch_labels = next(iter(dataloader))
print(f"Image batch [B, C, H, W]: {batch_images.shape}")
print(f"Label batch [B]: {batch_labels.shape}")


Image batch [B, C, H, W]: torch.Size([4, 3, 224, 224])
Label batch [B]: torch.Size([4])


## Split data and create `DataLoader`s

Split the dataset before creating loaders so training, validation, and test samples remain separate. Shuffle training data each epoch, but keep validation and test order stable for repeatable evaluation.

For batch size $B=32$, each loader returns images with shape $[B, C, H, W] = [B, 3, 224, 224]$ and labels with shape $[B]$. The final batch may be smaller than 32.


In [10]:
# Split sample indices into 70% training, 15% validation, and 15% test data.
train_size: int = int(0.70 * len(transformed_dataset))
validation_size: int = int(0.15 * len(transformed_dataset))
test_size: int = len(transformed_dataset) - train_size - validation_size

train_dataset, validation_dataset, test_dataset = random_split(
    transformed_dataset,
    [train_size, validation_size, test_size],
    generator=torch.Generator().manual_seed(42),  # Reproduce the same partition.
)

print(f"Training: {len(train_dataset)} images")
print(f"Validation: {len(validation_dataset)} images")
print(f"Test: {len(test_dataset)} images")


Training: 5732 images
Validation: 1228 images
Test: 1229 images


In [11]:
# Shuffle only training samples; evaluation order does not affect metrics.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [12]:
# Each iteration returns images [B, C, H, W] and labels [B].
for batch_images, batch_labels in train_loader:
    print(f"Image batch [B, C, H, W]: {batch_images.shape}")
    print(f"Label batch [B]: {batch_labels.shape}")
    break  # One batch is enough for this inspection.


Image batch [B, C, H, W]: torch.Size([32, 3, 224, 224])
Label batch [B]: torch.Size([32])


In [13]:
# next(iter(...)) is the shortest way to inspect exactly one batch.
batch_images, batch_labels = next(iter(train_loader))
print(f"Image batch [B, C, H, W]: {batch_images.shape}")
print(f"Label batch [B]: {batch_labels.shape}")


Image batch [B, C, H, W]: torch.Size([32, 3, 224, 224])
Label batch [B]: torch.Size([32])


### Count batches across one complete epoch

`len(train_loader)` is the number of batches, while `len(train_dataset)` is the number of samples. With `drop_last=False` (the default), the final batch may be smaller than the configured batch size:

$$
N_{\text{batches}} = \left\lceil \frac{N_{\text{samples}}}{B} \right\rceil.
$$

During iteration, images have shape $[B, C, H, W]$ and labels have shape $[B]$. The assertions verify that one epoch neither loses nor duplicates samples.


In [ ]:
batch_count: int = 0
total_images: int = 0

# One epoch visits every training sample once, one [B, C, H, W] batch at a time.
for batch_images, batch_labels in train_loader:
    batch_count += 1
    total_images += len(batch_images)

    # Show the final three batches without hard-coding their batch numbers.
    if batch_count > len(train_loader) - 3:
        print(
            f"Batch {batch_count}/{len(train_loader)}: "
            f"images {tuple(batch_images.shape)}, labels {tuple(batch_labels.shape)}"
        )

print()
print(f"Total batches in one epoch: {batch_count}")
print(f"Total images seen: {total_images}")
assert batch_count == len(train_loader)
assert total_images == len(train_dataset)


## Avoid expensive work inside `__getitem__()`

Loading **one requested sample** inside `__getitem__()` is correct—the flower dataset does exactly that. Loading and parsing the **entire source dataset** on every call is not.

A `DataLoader` calls `__getitem__()` once per sample, so the class below would re-read the complete CSV for every row in every epoch. Real tabular datasets should load the table once in `__init__()` and return one already-loaded row from `__getitem__()`. The class is intentionally named `BadDataset` and is not instantiated.


In [ ]:
import pandas as pd


class BadDataset(Dataset[pd.Series]):
    """Demonstrate the cost of reparsing an entire table for every sample.

    This class is intentionally inefficient and should not be used for training.
    It also omits ``__len__()`` because it is only an anti-pattern illustration.
    """

    def __getitem__(self, index: int) -> pd.Series:
        """Re-read the full CSV, then return one row—an intentional anti-pattern."""
        # Every access repeats disk I/O and CSV parsing, even for adjacent rows.
        table: pd.DataFrame = pd.read_csv("huge_file.csv")
        return table.iloc[index]
